# 🎙️ Real-Time Speech Emotion Recognition using MFCC, CNN-BiLSTM & Attention (Google Colab)

**Objective:** Classify the emotion expressed in a speech clip — neutral, calm, happy, sad, angry,
fearful, disgust, or surprised — from audio alone.

**Dataset:** [RAVDESS Emotional Speech Audio (Kaggle)](https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio)
- 1,440 WAV files, 24 professional actors (12 male, 12 female)
- 8 emotions, each spoken at two intensities, two statements, two repetitions
- Emotion is encoded directly in the filename, e.g. `03-01-06-01-02-01-12.wav`

**This single notebook covers the full pipeline, base through advanced:**

`Load audio → EDA → MFCC + Mel spectrogram extraction → Data augmentation → CNN baseline →
CNN-BiLSTM + Attention → Evaluation (confusion matrix, F1 per emotion) → Attention
visualization → Real-time microphone inference → Export artifacts for deployment`

Target resume title: **Real-Time Speech Emotion Recognition using MFCC, CNN-BiLSTM & Attention**


## 0. Get the dataset into Colab

**Option A — Kaggle API (recommended):**
1. Go to [kaggle.com](https://www.kaggle.com) → your profile → *Settings* → *API* → **Create New Token** (downloads `kaggle.json`).
2. Run the cell below and upload `kaggle.json` when prompted.

**Option B — Manual upload:** download the zip from the
[Kaggle dataset page](https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio)
yourself and upload it with `files.upload()`, then unzip it, skipping Option A.


In [ ]:
!pip install -q librosa soundfile

# OPTION A: Kaggle API download
from google.colab import files
import os

if not os.path.exists("ravdess"):
    print("Upload your kaggle.json (Kaggle -> Settings -> API -> Create New Token)")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !pip install -q kaggle
    !kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio
    !unzip -oq ravdess-emotional-speech-audio.zip -d ravdess


In [ ]:
# OPTION B: Manual upload (skip if you used Option A above)
# uploaded = files.upload()  # choose the Kaggle zip from your computer, then:
# !unzip -oq *.zip -d ravdess


## A. Parse labels from filenames & build a file index

In [ ]:
import os
import glob
import pandas as pd

EMOTION_MAP = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised",
}

def build_file_index(root_dir: str) -> pd.DataFrame:
    rows = []
    for path in glob.glob(os.path.join(root_dir, "**", "*.wav"), recursive=True):
        filename = os.path.basename(path)
        parts = filename.replace(".wav", "").split("-")
        if len(parts) != 7:
            continue
        modality, vocal_channel, emotion_code, intensity, statement, repetition, actor = parts
        rows.append({
            "path": path,
            "emotion": EMOTION_MAP.get(emotion_code, "unknown"),
            "intensity": "strong" if intensity == "02" else "normal",
            "actor": int(actor),
            "sex": "female" if int(actor) % 2 == 0 else "male",
        })
    return pd.DataFrame(rows)

df = build_file_index("ravdess")
print("Total audio files found:", len(df))
df.head()


## B–C. Data Understanding & EDA

In [ ]:
print(df["emotion"].value_counts())
print()
print(df["sex"].value_counts())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(8,4))
sns.countplot(data=df, x="emotion", order=sorted(df["emotion"].unique()), ax=ax)
ax.set_title("Emotion Class Distribution")
plt.xticks(rotation=30)
plt.show()


In [ ]:
import librosa
import librosa.display
import numpy as np

sample_path = df[df["emotion"] == "happy"].iloc[0]["path"]
y, sr = librosa.load(sample_path, sr=22050)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
librosa.display.waveshow(y, sr=sr, ax=axes[0])
axes[0].set_title("Waveform - Example 'happy' clip")

mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
img = librosa.display.specshow(mel_spec_db, sr=sr, x_axis="time", y_axis="mel", ax=axes[1])
axes[1].set_title("Mel Spectrogram - Example 'happy' clip")
fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
plt.tight_layout()
plt.show()


## D. Audio Preprocessing & Feature Extraction (MFCC)

**What is an MFCC?** A Mel-Frequency Cepstral Coefficient is a compact numeric summary of the
short-term power spectrum of a sound, on the Mel scale (which mimics how the human ear perceives
pitch — more resolution at low frequencies, less at high). Speech carries most of its emotional
and phonetic information in a fairly small number of these coefficients (commonly 13–40), which
is why MFCCs are the standard first feature for speech tasks.

We pad/truncate every clip to a fixed number of time frames so all feature arrays have the same
shape (required for batching into a neural network) — this handles the "different audio lengths"
problem.


In [ ]:
SAMPLE_RATE = 22050
N_MFCC = 40
MAX_LEN = 130  # ~3 seconds of frames at hop_length=512, sr=22050 - covers nearly all RAVDESS clips

def extract_mfcc_from_array(y, sr=SAMPLE_RATE, n_mfcc=N_MFCC, max_len=MAX_LEN):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    if mfcc.shape[1] < max_len:
        mfcc = np.pad(mfcc, ((0, 0), (0, max_len - mfcc.shape[1])), mode="constant")
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc

def extract_features(path: str, sr: int = SAMPLE_RATE, n_mfcc: int = N_MFCC, max_len: int = MAX_LEN):
    y, _ = librosa.load(path, sr=sr)
    y, _ = librosa.effects.trim(y, top_db=25)  # trim leading/trailing silence
    return extract_mfcc_from_array(y, sr, n_mfcc, max_len)

# Quick sanity check
sample_mfcc = extract_features(sample_path)
print("MFCC shape:", sample_mfcc.shape)

plt.figure(figsize=(8, 4))
librosa.display.specshow(sample_mfcc, x_axis="time")
plt.colorbar()
plt.title("MFCC - Example 'happy' clip")
plt.show()


## E. Data Augmentation (applied to training data only, after the split, to avoid leakage)

In [ ]:
def add_noise(y, noise_factor=0.005):
    noise = np.random.randn(len(y))
    return y + noise_factor * noise

def pitch_shift(y, sr, n_steps=2):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def time_stretch(y, rate=1.1):
    return librosa.effects.time_stretch(y, rate=rate)

def augment_and_extract(path: str, sr: int = SAMPLE_RATE):
    """Returns a list of MFCC feature arrays: original + 3 augmented versions."""
    y, _ = librosa.load(path, sr=sr)
    y, _ = librosa.effects.trim(y, top_db=25)

    variants = [
        y,
        add_noise(y),
        pitch_shift(y, sr, n_steps=int(np.random.choice([-2, 2]))),
        time_stretch(y, rate=float(np.random.choice([0.9, 1.1]))),
    ]
    return [extract_mfcc_from_array(v, sr=sr) for v in variants]


## F. Build the dataset (extract features for every file) & split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Split by ACTOR first (not by clip) so the same actor's voice never appears in
# both train and test - this prevents the model from just memorizing voices.
actors = df["actor"].unique()
train_actors, test_actors = train_test_split(actors, test_size=0.2, random_state=42)

train_df = df[df["actor"].isin(train_actors)].reset_index(drop=True)
test_df = df[df["actor"].isin(test_actors)].reset_index(drop=True)

print("Train clips:", len(train_df), " Test clips:", len(test_df))


In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(df["emotion"])
num_classes = len(label_encoder.classes_)
print("Classes:", list(label_encoder.classes_))

# Test set: original clips only, no augmentation
X_test, y_test = [], []
for _, row in test_df.iterrows():
    X_test.append(extract_features(row["path"]))
    y_test.append(row["emotion"])

X_test = np.array(X_test)
y_test_enc = label_encoder.transform(y_test)
print("X_test shape:", X_test.shape)


In [ ]:
# Training set: original + augmented versions of every clip
X_train, y_train = [], []
for _, row in train_df.iterrows():
    for feat in augment_and_extract(row["path"]):
        X_train.append(feat)
        y_train.append(row["emotion"])

X_train = np.array(X_train)
y_train_enc = label_encoder.transform(y_train)
print("X_train shape (after augmentation):", X_train.shape)


In [ ]:
# Normalize features using TRAIN statistics only (no leakage)
mean, std = X_train.mean(), X_train.std()
X_train_norm = (X_train - mean) / (std + 1e-8)
X_test_norm = (X_test - mean) / (std + 1e-8)

# Add channel dimension for Conv2D: (n_mfcc, max_len, 1)
X_train_cnn = X_train_norm[..., np.newaxis]
X_test_cnn = X_test_norm[..., np.newaxis]

from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train_enc, num_classes=num_classes)
y_test_cat = to_categorical(y_test_enc, num_classes=num_classes)

print(X_train_cnn.shape, X_test_cnn.shape)


## G. CNN Baseline

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3,3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3,3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3,3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation="softmax"),
    ])
    return model

cnn_model = build_cnn(X_train_cnn.shape[1:], num_classes)
cnn_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
cnn_model.summary()


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
]

cnn_history = cnn_model.fit(
    X_train_cnn, y_train_cat,
    validation_data=(X_test_cnn, y_test_cat),
    epochs=60, batch_size=32, callbacks=callbacks, verbose=1,
)


## H. CNN-BiLSTM with an Attention layer (the resume-worthy upgrade)

Instead of stopping at a plain CNN-BiLSTM, we add an **attention layer** on top of the BiLSTM
output. Rather than just taking the final hidden state, attention looks at every time step and
learns a relevance weight for each one, then combines them into a single context vector. This is
what lets the model — and us — answer *"which moments of the clip mattered most for this
prediction?"* later in the notebook.


In [ ]:
class AdditiveAttention(layers.Layer):
    """A simple additive (Bahdanau-style) attention pooling layer.

    Takes a (batch, time_steps, features) sequence and returns a single
    (batch, features) context vector, weighted by a learned relevance score
    per time step. Stores the last computed weights on
    `self.last_attention_weights` for explainability.
    """
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.last_attention_weights = None

    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.W = self.add_weight(shape=(feature_dim, self.units), initializer="glorot_uniform", trainable=True, name="att_W")
        self.b = self.add_weight(shape=(self.units,), initializer="zeros", trainable=True, name="att_b")
        self.u = self.add_weight(shape=(self.units, 1), initializer="glorot_uniform", trainable=True, name="att_u")

    def call(self, inputs):
        score = tf.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)   # (batch, time, units)
        score = tf.tensordot(score, self.u, axes=1)                       # (batch, time, 1)
        weights = tf.nn.softmax(score, axis=1)                            # (batch, time, 1)
        self.last_attention_weights = weights
        context = tf.reduce_sum(inputs * weights, axis=1)                 # (batch, features)
        return context

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config


def build_cnn_bilstm_attention(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, (3,3), activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3,3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    # Reshape (freq, time, channels) -> (time, freq*channels) so LSTM reads across time
    shape = x.shape
    x = layers.Permute((2, 1, 3))(x)
    x = layers.Reshape((shape[2], shape[1] * shape[3]))(x)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)

    x = AdditiveAttention(units=64, name="attention")(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inputs, outputs, name="cnn_bilstm_attention")

final_model = build_cnn_bilstm_attention(X_train_cnn.shape[1:], num_classes)
final_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
final_model.summary()


In [ ]:
final_history = final_model.fit(
    X_train_cnn, y_train_cat,
    validation_data=(X_test_cnn, y_test_cat),
    epochs=60, batch_size=32, callbacks=callbacks, verbose=1,
)


## I. Evaluation: Confusion Matrix & F1-score per emotion

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

def evaluate_model(model, X_test, y_test_enc, label_encoder, name):
    y_pred_proba = model.predict(X_test)
    y_pred = np.argmax(y_pred_proba, axis=1)

    print(f"=== {name} ===")
    print(classification_report(y_test_enc, y_pred, target_names=label_encoder.classes_))

    cm = confusion_matrix(y_test_enc, y_pred)
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return y_pred

cnn_pred = evaluate_model(cnn_model, X_test_cnn, y_test_enc, label_encoder, "CNN Baseline")


In [ ]:
final_pred = evaluate_model(final_model, X_test_cnn, y_test_enc, label_encoder, "CNN-BiLSTM + Attention")


In [ ]:
f1_scores = f1_score(y_test_enc, final_pred, average=None)
f1_df = pd.DataFrame({"emotion": label_encoder.classes_, "f1_score": f1_scores}).sort_values("f1_score")

plt.figure(figsize=(8,4))
sns.barplot(data=f1_df, x="f1_score", y="emotion", color="steelblue")
plt.title("F1-Score per Emotion - CNN-BiLSTM + Attention")
plt.xlim(0, 1)
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(cnn_history.history["val_accuracy"], label="CNN val accuracy")
plt.plot(final_history.history["val_accuracy"], label="CNN-BiLSTM+Attention val accuracy")
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy")
plt.title("CNN vs CNN-BiLSTM+Attention - Validation Accuracy over Training")
plt.legend()
plt.show()


## J. Attention visualization — why did the model predict this emotion?

We call the model directly (eager execution, not `.predict()`) on a single clip so the attention
layer's `last_attention_weights` reflects that exact input, then plot the weight per time frame
alongside the MFCC itself.


In [ ]:
def visualize_attention(path_or_array, true_label=None, is_path=True):
    if is_path:
        mfcc = extract_features(path_or_array)
    else:
        mfcc = extract_mfcc_from_array(path_or_array)

    x = ((mfcc - mean) / (std + 1e-8))[np.newaxis, ..., np.newaxis]

    proba = final_model(x, training=False).numpy()[0]
    pred_idx = np.argmax(proba)
    pred_label = label_encoder.inverse_transform([pred_idx])[0]

    attn_layer = final_model.get_layer("attention")
    weights = attn_layer.last_attention_weights.numpy()[0, :, 0]

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    librosa.display.specshow(mfcc, x_axis="time", ax=axes[0])
    axes[0].set_title(f"MFCC — predicted: {pred_label} ({proba[pred_idx]*100:.1f}%)"
                       + (f" | true: {true_label}" if true_label else ""))

    axes[1].plot(weights)
    axes[1].fill_between(range(len(weights)), weights, alpha=0.3)
    axes[1].set_title("Attention weight per time frame (which moments mattered most)")
    axes[1].set_xlabel("Frame")
    plt.tight_layout()
    plt.show()

# Try it on a random test clip
sample_row = test_df.sample(1).iloc[0]
visualize_attention(sample_row["path"], true_label=sample_row["emotion"])


## K. Real-time microphone inference (inside Colab)

Colab can't access your OS microphone directly, but it can run a small JavaScript snippet in the
browser tab to record from your mic and hand the audio back to Python. Run the cell below, allow
microphone access when prompted, speak a short sentence, then it predicts the emotion.


In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
var recorder, recordedChunks = [];

async function record(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  recordedChunks = [];
  recorder.ondataavailable = e => recordedChunks.push(e.data);
  recorder.start();
  await sleep(seconds * 1000);
  recorder.stop();
  await sleep(500);
  const blob = new Blob(recordedChunks);
  const arrayBuffer = await blob.arrayBuffer();
  const bytes = new Uint8Array(arrayBuffer);
  let binary = '';
  for (let i = 0; i < bytes.byteLength; i++) binary += String.fromCharCode(bytes[i]);
  return btoa(binary);
}
"""

def record_from_mic(seconds=4):
    display(Javascript(RECORD_JS))
    print(f"Recording for {seconds} seconds - speak now...")
    audio_b64 = eval_js(f"record({seconds})")
    audio_bytes = b64decode(audio_b64)
    with open("mic_recording.webm", "wb") as f:
        f.write(audio_bytes)
    print("Recording saved.")
    return "mic_recording.webm"

# Run this cell, allow mic access, then speak:
mic_path = record_from_mic(seconds=4)


In [ ]:
# librosa can't load webm directly - convert to wav first (ffmpeg is preinstalled on Colab)
!ffmpeg -y -loglevel error -i mic_recording.webm mic_recording.wav

visualize_attention("mic_recording.wav")


## L. Export artifacts for deployment (API / Streamlit app)

Save `emotion_model.keras`, `label_encoder.pkl`, and `norm_stats.pkl` — these are exactly what a
`predict.py`/FastAPI/Streamlit deployment would load to serve predictions without retraining.


In [ ]:
import pickle

final_model.save("emotion_model.keras")

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

with open("norm_stats.pkl", "wb") as f:
    pickle.dump({"mean": float(mean), "std": float(std), "n_mfcc": N_MFCC, "max_len": MAX_LEN, "sr": SAMPLE_RATE}, f)

with open("metrics.json", "w") as f:
    import json
    json.dump({
        "cnn_baseline": classification_report(y_test_enc, cnn_pred, target_names=label_encoder.classes_, output_dict=True),
        "cnn_bilstm_attention": classification_report(y_test_enc, final_pred, target_names=label_encoder.classes_, output_dict=True),
    }, f, indent=2)

from google.colab import files
files.download("emotion_model.keras")
files.download("label_encoder.pkl")
files.download("norm_stats.pkl")
files.download("metrics.json")
print("Artifacts saved and downloaded. Move them into your project's models/ folder.")


## What's next

This single notebook now covers the full "Must-have" list: MFCC extraction, data augmentation,
a CNN baseline, a CNN-BiLSTM model with an attention mechanism, per-emotion F1/confusion-matrix
evaluation, attention-weight visualization, and real-time microphone inference — everything
needed for:

**Real-Time Speech Emotion Recognition using MFCC, CNN-BiLSTM & Attention**

A full production project (FastAPI + Streamlit app with live mic recording + tests + README)
that consumes these exact exported artifacts is available as a separate ready-to-run repository.
